In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Household and Population

In [ ]:
def hh_and_pop(data1, data3, tag='PSRC Region'):
    # Merge data
    merge_per_hh_1 = pd.merge(data1['Person'][['pwtyp', 'psexpfac', 'pwpcl', 'pwaudist','pstyp', 'pspcl', 'psaudist', 'hhno', 'ptpass']],
                            data1['Household'][['hhtaz', 'hhparcel', 'hhno']],
                            on = 'hhno')
    merge_per_hh_3 = pd.merge(data3['Person'][['pwtyp', 'psexpfac', 'pwpcl', 'pwaudist','pstyp', 'pspcl', 'psaudist', 'hhno', 'ptpass']],
                            data3['Household'][['hhtaz', 'hhparcel', 'hhno']],
                            on = 'hhno')
    df_summary = pd.DataFrame(columns=['DaysimOutputs', f'{survey_year}Survey', 
                                    f'Difference (DaysimOutputs - {survey_year}Survey)', 
                                    f'% Difference (DaysimOutputs - {survey_year}Survey)'],
                            index=['Total Persons', 'Total Households', 
                                    'Average Household Size', 'Average Trips Per Person', 'Average Trip Length',
                                    'Vehicle Miles per Person', 'Average Distance to Work (Non-Home)', 'Average Distance to School (Non-Home)'])
    df_summary = df_summary.astype('float64')

    trip_ok_1 = data1['Trip'][['travdist', 'trexpfac', 'dorp']].query('travdist > 0 and travdist < 200')
    trip_ok_3 = data3['Trip'][['travdist', 'trexpfac', 'dorp']].query('travdist > 0 and travdist < 200')

    ##Basic Summaries
    #Total Households, Persons, and Trips
    tp1 = data1['Person']['psexpfac'].sum()  # total persons
    tp3 = data3['Person']['psexpfac'].sum()
    th1 = data1['Household']['hhexpfac'].sum()  # total households
    th3 = data3['Household']['hhexpfac'].sum()
    ttr1 = trip_ok_1['trexpfac'].sum()  # total trips
    ttr3 = trip_ok_3['trexpfac'].sum()
    ahhs1 = tp1 / th1  # average household size
    ahhs3 = tp3 / th3
    ntr1 = ttr1 / tp1  # average number of trips per person
    ntr3 = ttr3 / tp3
    atl1 = weighted_average(trip_ok_1, 'travdist', 'trexpfac', grouper=None)  # average trip length
    atl3 = weighted_average(trip_ok_3, 'travdist', 'trexpfac', grouper=None)
    driver_trips1 = trip_ok_1[['dorp', 'travdist', 'trexpfac']].query('dorp == "Driver"')  # vehicle miles (unweighted)
    driver_trips3 = trip_ok_3[['dorp', 'travdist', 'trexpfac']].query('dorp == "Driver"')
    vmpp1sp = (driver_trips1['travdist'].multiply(driver_trips1['trexpfac'])).sum()  # weighted vehicle miles
    vmpp3sp = (driver_trips3['travdist'].multiply(driver_trips3['trexpfac'])).sum()
    vmpp1 = vmpp1sp / tp1  # vehicle miles per person
    vmpp3 = vmpp3sp / tp3

    #Work Location
    wrkrs1 = merge_per_hh_1[['pwtyp', 'hhtaz', 'psexpfac', 'pwpcl', 'pwaudist', 'hhparcel']].\
        query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wrkrs3 = merge_per_hh_3[['pwtyp', 'hhtaz', 'psexpfac', 'pwpcl', 'pwaudist', 'hhparcel']].\
        query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wrkr_1_hzone = pd.merge(wrkrs1, taz_subarea, left_on = 'hhtaz', right_on = 'TAZ')
    wrkr_3_hzone = pd.merge(wrkrs3, taz_subarea, left_on = 'hhtaz', right_on = 'TAZ')
    # only take those in-person workers: usual work location parcel != home location parcel
    workers_1 = wrkr_1_hzone.query('pwpcl != hhparcel and pwaudist > 0 and pwaudist < 200').copy()
    workers_3 = wrkr_3_hzone.query('pwpcl != hhparcel and pwaudist > 0 and pwaudist < 200').copy()
    workers_1['Share (%)'] = workers_1['psexpfac'] / workers_1['psexpfac'].sum()
    workers_3['Share (%)'] = workers_3['psexpfac'] / workers_3['psexpfac'].sum()
    workers1_avg_dist = weighted_average(workers_1, 'pwaudist', 'psexpfac')
    workers3_avg_dist = weighted_average(workers_3, 'pwaudist', 'psexpfac')
    #School Location
    st1 = merge_per_hh_1[['pstyp', 'hhtaz', 'psexpfac', 'pspcl', 'psaudist', 'hhparcel']].\
        query('pstyp == "Full-Time Student" or pstyp == "Part-Time Student"')
    st3 = merge_per_hh_3[['pstyp', 'hhtaz', 'psexpfac', 'pspcl', 'psaudist', 'hhparcel']].\
        query('pstyp == "Full-Time Student" or pstyp == "Part-Time Student"')
    st_1_hzone = pd.merge(st1, taz_subarea, 'outer', left_on = 'hhtaz', right_on = 'TAZ')
    st_3_hzone = pd.merge(st3, taz_subarea, 'outer', left_on = 'hhtaz', right_on = 'TAZ')
    # only take those in-person students: usual school/university location parcel != home location parcel
    students_1 = st_1_hzone.query('pspcl != hhparcel and psaudist > 0 and psaudist < 200').copy()
    students_3 = st_3_hzone.query('pspcl != hhparcel and psaudist > 0 and psaudist < 200').copy()
    students_1['Share (%)'] = students_1['psexpfac'] / students_1['psexpfac'].sum()
    students_3['Share (%)'] = students_3['psexpfac'] / students_3['psexpfac'].sum()
    students1_avg_dist = weighted_average(students_1, 'psaudist', 'psexpfac')
    students3_avg_dist = weighted_average(students_3, 'psaudist', 'psexpfac')

    df_summary.loc[:, ['DaysimOutputs', f'{survey_year}Survey']] = [[tp1, tp3],
                                                                [th1, th3],
                                                                [ahhs1, ahhs3],
                                                                [ntr1, ntr3],
                                                                [atl1, atl3],
                                                                [vmpp1, vmpp3],
                                                                [workers1_avg_dist, workers3_avg_dist],
                                                                [students1_avg_dist, students3_avg_dist]]
    df_summary = get_differences(df_summary, 'DaysimOutputs', f'{survey_year}Survey', 1)

    # table
    display(df_summary.style.format({'DaysimOutputs': '{:.1f}', 
                                    f'{survey_year}Survey': '{:.1f}', 
                                    f'Difference (DaysimOutputs - {survey_year}Survey)': '{:.1f}',
                                    f'% Difference (DaysimOutputs - {survey_year}Survey)': '{:.1f}%'}))

    # figure
    fig = px.bar(
        df_summary.loc[['Average Household Size',
                        'Average Trips Per Person',
                        'Average Trip Length',
                        'Vehicle Miles per Person',
                        'Average Distance to Work (Non-Home)',
                        'Average Distance to School (Non-Home)'], :].reset_index(),
        x='index',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Daysim Outputs vs Survey Summary ({tag})',
        labels={'index': 'Metric', 'value': 'Value', 'variable': 'Source'},
    )
    fig.update_layout(xaxis_title='', yaxis_title='', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
hh_and_pop(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
hh_and_pop(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Population by District

In [ ]:
data_daysim['Person'] = data_daysim['Person'].merge(data_daysim['Household'][['hhno', 'hhtaz']], on='hhno', how='left')
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(data_fullsurvey['Household'][['hhno', 'hhtaz']], on='hhno', how='left')
data_daysim['Person'] = get_subarea(_data=data_daysim['Person'], taz_subarea=taz_subarea, taz_colname='hhtaz')
data_fullsurvey['Person'] = get_subarea(_data=data_fullsurvey['Person'], taz_subarea=taz_subarea, taz_colname='hhtaz')

result_daysim = data_daysim['Person'].groupby(by='DistrictFlowName')['psexpfac'].sum()
result_fullsurvey = data_fullsurvey['Person'].groupby(by='DistrictFlowName')['psexpfac'].sum()
result_daysim.loc['Total'] = result_daysim.sum()
result_fullsurvey.loc['Total'] = result_fullsurvey.sum()
# Concatenate the two results into a single DataFrame for comparison
comparison_df = pd.concat(
    [result_daysim.rename('DaysimOutputs'), result_fullsurvey.rename(f'{survey_year}Survey')],
    axis=1
)
comparison_df = comparison_df.loc[district_flow_name.values()]
# table
display(comparison_df.style.format('{:,.0f}'))

## Workers by District

In [ ]:
from collections import OrderedDict

def ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='worker', pptyp_label='Worker'):
    """
    pptyp: person type that you want to query, 'worker', or 'student'.
    pptyp_label: 'Worker', 'Student'.
    """
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey'
    #Workers, and students by District
    workers_per_taz_1 = data1['Person'][[f'p{pptyp[0]}taz', 'psexpfac']].groupby(f'p{pptyp[0]}taz').sum()['psexpfac']
    workers_per_taz_3 = data3['Person'][[f'p{pptyp[0]}taz', 'psexpfac']].groupby(f'p{pptyp[0]}taz').sum()['psexpfac']
    workers_per_taz = pd.DataFrame.from_dict(OrderedDict(((f'Number of {pptyp_label}s (' + name1 + ')', workers_per_taz_1), 
                                                            (f'Number of {pptyp_label}s (' + name3 + ')', workers_per_taz_3))))
    workers_per_taz_district = pd.merge(workers_per_taz, taz_subarea, left_index = True, right_on = 'TAZ')
    workers_per_district = workers_per_taz_district[[f'Number of {pptyp_label}s (' + name1 + ')', 
                                                     f'Number of {pptyp_label}s (' + name3 + ')', 'DistrictFlowName']].groupby('DistrictFlowName').sum()
    workers_per_district = get_differences(workers_per_district, f'Number of {pptyp_label}s (' + name1 + ')', 
                                                                 f'Number of {pptyp_label}s (' + name3 + ')', 
                                                                 0) 
    workers_per_district = workers_per_district.loc[district_flow_name.values()]
    display(workers_per_district.style.format({f'Number of {pptyp_label}s (' + name1 + ')': '{:,.0f}',
                                               f'Number of {pptyp_label}s (' + name3 + ')': '{:,.0f}',
                                               f'Difference (Number of {pptyp_label}s ({name1}) - Number of {pptyp_label}s ({name3}))': '{:,.0f}',
                                               f'% Difference (Number of {pptyp_label}s ({name1}) - Number of {pptyp_label}s ({name3}))': '{:,.1f}%', }))

    fig = px.bar(
        workers_per_district.reset_index(),
        x='DistrictFlowName',
        y=[f'Number of {pptyp_label}s (' + name1 + ')', f'Number of {pptyp_label}s (' + name3 + ')'],
        barmode='group',
        title=f'{pptyp_label}s by District',
        labels={'value': f'Number of {pptyp_label}s', 'variable': 'Source', 'DistrictFlowName': 'District'}
    )
    fig.update_layout(xaxis_title='', 
                      yaxis_title=f'Number of {pptyp_label}s', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='worker', pptyp_label='Worker')

## Students by District

In [ ]:
ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='student', pptyp_label='Student')

## Population by Person Type

In [ ]:
def pop_by_pptyp(person_daysim, person_fullsurvey):
    result_daysim = person_daysim.groupby(by='pptyp')['psexpfac'].sum()
    result_fullsurvey = person_fullsurvey.groupby(by='pptyp')['psexpfac'].sum()
    result_daysim.loc['Total'] = result_daysim.sum()
    result_fullsurvey.loc['Total'] = result_fullsurvey.sum()
    # Concatenate the two results into a single DataFrame for comparison
    comparison_df = pd.concat(
        [result_daysim.rename('DaysimOutputs'), result_fullsurvey.rename(f'{base_year}Survey')],
        axis=1
    )

    comparison_df.index.name = 'Person Type'
    # table
    display(comparison_df.loc[ptype_cat.values()].style.format('{:,.0f}'))

In [ ]:
pop_by_pptyp(data_daysim['Person'], data_fullsurvey['Person'])

In [ ]:
pop_by_pptyp(data_daysim_bkr['Person'], data_fullsurvey_bkr['Person'])

## Population by Person Type and District

In [ ]:
pop_by_type_and_district = data_daysim['Person'].groupby(['pptyp', 'DistrictFlowName'])['psexpfac'].sum().unstack('DistrictFlowName')
pop_by_type_and_district.index.name = 'Person Type'
display(pop_by_type_and_district.loc[ptype_cat.values()].style.format('{:,.0f}'))

In [ ]:
pop_by_type_and_district = data_fullsurvey['Person'].groupby(['pptyp', 'DistrictFlowName'])['psexpfac'].sum().unstack('DistrictFlowName')
pop_by_type_and_district.index.name = 'Person Type'
display(pop_by_type_and_district.loc[ptype_cat.values()].style.format('{:,.0f}'))

## Transit Pass Ownership

In [ ]:
def transit_pass_ownership(data1, data2, data3, tag='PSRC Region'):
    ttp1 = data1['Person']['ptpass'].multiply(data1['Person']['psexpfac']).sum()
    ttp2 = data2['Person'].loc[data2['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ttp3 = data3['Person'].loc[data3['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ppp1 = ttp1 / get_total(data1['Person']['psexpfac'])
    ppp3 = ttp3 / get_total(data3['Person']['psexpfac'])
    tpass = pd.DataFrame(index = ['Total Passes', 'Passes per Person'])
    tpass['DaysimOutputs'] = [ttp1, ppp1]
    tpass[f'{survey_year}Survey'] = [ttp3, ppp3]
    tpass = get_differences(tpass, 'DaysimOutputs', f'{survey_year}Survey', [0, 2])
    # table
    display(tpass.style.format('{:,.1f}'))
    # bar plot
    fig = px.bar(
        tpass.loc[['Total Passes'], ['DaysimOutputs', f'{survey_year}Survey']].reset_index(),
        x='index',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Total Transit Passes Comparison ({tag})',
        labels={'index': 'Metric', 'value': 'Total Passes', 'variable': 'Source'},
    )
    fig.update_layout(xaxis_title='', 
                      yaxis_title='Total Passes', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
transit_pass_ownership(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
transit_pass_ownership(data1=data_daysim_bkr, data2=data_survey_bkr, data3=data_fullsurvey_bkr, tag='BKR')

## Automobile Ownership

In [ ]:
def auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):
    #Auto Ownership
    # only includes stats from data1 (model outputs) and data3 (full survey), as data2 (daysim-formatted) removed many records
    ao1 = 100 * data1['Household'][['hhvehs','hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data1['Household']['hhexpfac'].sum()
    veh3_ok = data3['Household'].query('hhvehs >= 0')
    ao3 = 100 * veh3_ok[['hhvehs','hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data3['Household']['hhexpfac'].sum()
    ao3 = ao3.reset_index()
    ao3.loc[(ao3['hhvehs']==4), 'hhexpfac'] = ao3.loc[(ao3['hhvehs']>=4), 'hhexpfac'].sum()
    ao3 = ao3[ao3['hhvehs'].isin([0, 1, 2, 3, 4])]
    ao3.set_index('hhvehs', inplace=True)
    for i in range(5, len(ao1)):
        ao1[4] = ao1[4] + ao1[i]
        ao1 = ao1.drop([i])
    for i in range(5, len(ao3)):
        ao3[4] = ao3[4] + ao3[i]
        ao3 = ao3.drop([i])

    ao = pd.DataFrame()

    # read in ACS dataset
    if tag == 'PSRC Region':
        acs_data_ = acs_data
    else:
        acs_data_ = acs_data_bkr
    autos= pd.read_excel(acs_data_,sheet_name = 'AutosTotal', engine = 'openpyxl')
    acs_auto_share = pd.DataFrame(autos['Share'] * 100).dropna(inplace=False)

    ao['Percent of Households (DaysimOutputs)'] = ao1
    ao[f'Percent of Households ({survey_year}Survey)'] = ao3
    ao['Percent of Households (ACS)'] = acs_auto_share 
    ao = get_differences_wt_fullsurvey(ao, 'Percent of Households (DaysimOutputs)', 
                                        f'Percent of Households ({survey_year}Survey)', 
                                        'Percent of Households (ACS)', 1, need_diff_percent=False)
    aonewcol = ['0', '1', '2', '3', '4+']
    ao['Number of Vehicles in Household'] = aonewcol
    ao = ao.reset_index()
    ao = ao.drop(columns = ['hhvehs'])
    ao = ao.set_index('Number of Vehicles in Household')
    # table
    display(ao.style.format('{:,.1f}%'))
    # plot
    fig = px.bar(
        ao.reset_index(),
        x='Number of Vehicles in Household',
        y=['Percent of Households (DaysimOutputs)', 
           f'Percent of Households ({survey_year}Survey)', 
           'Percent of Households (ACS)'],
        barmode='group',
        title=f'Automobile Ownership by Household ({tag})',
        labels={'value': 'Percent of Households', 'variable': 'Source', 'Number of Vehicles in Household': 'Number of Vehicles'}
    )
    fig.update_layout(yaxis_title='Percent of Households', 
                      xaxis_title='Number of Vehicles in Household', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
auto_ownership(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')